# 04 QC: Check on-target and off-target variants

This notebook checks whether variants from DeepVariant and Mutect2 fall inside the Agilent capture BED target regions.

For each processed sample and caller, it:

- loads the VEP report
- converts variant coordinates to BED-style intervals
- checks whether each variant overlaps a BED target interval
- labels each variant as on-target or off-target
- writes a per-variant target-status CSV
- writes an Excel summary workbook

## Setup
Set input folders, BED file, metrics file, and output folder.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

notebook_dir = Path("/home/donetski/Notebooks")
input_dir = notebook_dir / "InputFiles"
output_dir = notebook_dir / "OutputFiles"  / "04_qc_checking_on_target"
output_dir.mkdir(parents=True, exist_ok=True)

metrics_path = input_dir / "sequencing_metrics_merged_with_grouplabels.csv"
bed_path = input_dir / "3454741_Covered.bed.txt"
deepvariant_dir = input_dir / "deepvariant_vep"
mutect2_dir = input_dir / "mutect2_vep"

output_prefix = "04"

## Settings

Choose which runs and callers to process.

Use `All` for `runs_to_process` to process all runs.

In [ ]:
sample_limit = None
excel_preview_n_samples = 10

# runs_to_process = ["Run1"]
# runs_to_process = ["Run2", "Run3"]
runs_to_process = "all"

callers = ["DeepVariant", "Mutect2"]

metrics_required_cols = ["SAMPLE", "Group"]

vep_required_cols = [
    "Sample.ID",
    "Chr",
    "Start",
    "REF",
    "ALT",
    "Gene",
    "Variant.Class",
    "Variant.Consequence",
    "Sample.Depth",
    "Sample.AltDepth",
    "Sample.AltFrac",
]

metrics_qc_cols = [
    "MEAN_TARGET_COVERAGE",
    "MEDIAN_TARGET_COVERAGE",
    "MAX_TARGET_COVERAGE",
    "PCT_USABLE_BASES_ON_TARGET",
    "ZERO_CVG_TARGETS_PCT",
    "PCT_TARGET_BASES_20X",
    "PCT_TARGET_BASES_100X",
    "PCT_EXC_OFF_TARGET",
]

## Helper functions

These functions load BED regions, build expected VEP paths, convert variant coordinates, and label variants as on-target or off-target.

The overlap rule is BED-style interval overlap, not exact coordinate matching.

In [ ]:
def require_columns(df, required_cols, label):
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")


def selected_runs(value):
    if isinstance(value, str):
        return "all" if value.strip().lower() == "all" else [value.strip()]
    return [str(run).strip() for run in value]


def normalize_chrom(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().removesuffix(".0")
    return value if value.startswith("chr") else f"chr{value}"


def ref_len(ref):
    return 1 if pd.isna(ref) or str(ref).strip() in {"", ".", "-", "nan", "None"} else len(str(ref).strip())


def expected_vep_path(sample_id, caller):
    folders = {"DeepVariant": deepvariant_dir, "Mutect2": mutect2_dir}
    suffixes = {
        "DeepVariant": "deepvariant.norm.vep.report.csv",
        "Mutect2": "mutect2.norm.vep.report.csv",
    }
    return folders[caller] / f"{sample_id}.{suffixes[caller]}"


def load_bed_file(path):
    bed = pd.read_csv(path, sep="\t", header=None, dtype=str, low_memory=False)
    bed = bed[bed[0].astype(str).str.startswith("chr")].iloc[:, :4].copy()
    bed.columns = ["bed_chrom", "bed_start", "bed_end", "bed_region"]

    bed["bed_chrom"] = bed["bed_chrom"].apply(normalize_chrom)
    bed["bed_start"] = pd.to_numeric(bed["bed_start"], errors="coerce")
    bed["bed_end"] = pd.to_numeric(bed["bed_end"], errors="coerce")
    bed = bed.dropna(subset=["bed_chrom", "bed_start", "bed_end"]).copy()
    bed[["bed_start", "bed_end"]] = bed[["bed_start", "bed_end"]].astype(int)

    bed["bed_interval_id"] = (
        bed["bed_chrom"] + ":"
        + bed["bed_start"].astype(str) + "-"
        + bed["bed_end"].astype(str) + "_"
        + bed["bed_region"].astype(str)
    )

    return bed


def add_variant_intervals(df):
    df = df.copy()
    start = pd.to_numeric(df["Start"], errors="coerce")

    df["variant_chrom"] = df["Chr"].apply(normalize_chrom)
    df["variant_start"] = (start - 1).astype("Int64")
    df["variant_end"] = (start - 1 + df["REF"].apply(ref_len)).astype("Int64")

    return df


def label_one_variant(row, bed_by_chrom):
    chrom = row["variant_chrom"]
    start = row["variant_start"]
    end = row["variant_end"]

    result = {
        "on_target": False,
        "matched_bed_region": None,
        "matched_bed_interval_id": None,
        "matched_bed_start": None,
        "matched_bed_end": None,
        "num_overlapping_bed_intervals": 0,
        "distance_to_nearest_target": None,
    }

    if pd.isna(chrom) or pd.isna(start) or pd.isna(end) or chrom not in bed_by_chrom:
        return pd.Series(result)

    start, end = int(start), int(end)
    bed_chr = bed_by_chrom[chrom]

    hits = bed_chr[
        (start < bed_chr["bed_end"])
        & (end > bed_chr["bed_start"])
    ]

    if not hits.empty:
        hit = hits.iloc[0]
        result.update({
            "on_target": True,
            "matched_bed_region": hit["bed_region"],
            "matched_bed_interval_id": hit["bed_interval_id"],
            "matched_bed_start": hit["bed_start"],
            "matched_bed_end": hit["bed_end"],
            "num_overlapping_bed_intervals": len(hits),
            "distance_to_nearest_target": 0,
        })
    else:
        distances = np.where(
            end <= bed_chr["bed_start"],
            bed_chr["bed_start"] - end,
            np.where(start >= bed_chr["bed_end"], start - bed_chr["bed_end"], 0),
        )
        result["distance_to_nearest_target"] = int(distances.min())

    return pd.Series(result)

## Load metrics and BED target regions

The merged metrics file defines which samples belong to each run.

The BED file defines the capture target intervals used to label variants as on-target or off-target.

In [ ]:
metrics = pd.read_csv(metrics_path, low_memory=False)
require_columns(metrics, metrics_required_cols, "Merged metrics file")

metrics = metrics.copy()
metrics["sample_id"] = metrics["SAMPLE"].astype(str)
metrics["run"] = metrics["Group"].astype(str).str.strip()

runs_selected = selected_runs(runs_to_process)

if runs_selected != "all":
    allowed_runs = {run.lower() for run in runs_selected}
    metrics = metrics[metrics["run"].str.lower().isin(allowed_runs)].copy()

metrics = metrics.drop_duplicates(["run", "sample_id"]).reset_index(drop=True)

if sample_limit is not None:
    metrics = metrics.head(sample_limit).copy()

run_tag = "all_runs" if runs_selected == "all" else "_".join(runs_selected)

bed = load_bed_file(bed_path)
bed_by_chrom = {chrom: sub.copy() for chrom, sub in bed.groupby("bed_chrom")}

print("Runs included:", sorted(metrics["run"].unique()))
print("Samples included:", metrics["sample_id"].nunique())
print("BED intervals loaded:", len(bed))

## Label variants as on-target or off-target

For each sample and caller, this section loads the VEP report, adds variant intervals, checks overlap with BED intervals, and stores the labeled variant rows.

In [ ]:
all_variant_rows = []
problems = []

for i, sample in enumerate(metrics.itertuples(index=False), start=1):
    print(f"Processing sample {i}/{len(metrics)}: {sample.sample_id}",end="\r",)

    for caller in callers:
        vep_path = expected_vep_path(sample.sample_id, caller)

        if not vep_path.exists():
            problems.append({
                "run": sample.run,
                "sample_id": sample.sample_id,
                "caller": caller,
                "issue": "missing VEP report",
                "expected_file": str(vep_path),
            })
            continue

        try:
            try:
                vep = pd.read_csv(vep_path, low_memory=False)
            except UnicodeDecodeError:
                vep = pd.read_csv(vep_path, encoding="latin1", low_memory=False)

            require_columns(vep, vep_required_cols, str(vep_path))

            if vep.empty:
                problems.append({
                    "run": sample.run,
                    "sample_id": sample.sample_id,
                    "caller": caller,
                    "issue": "empty VEP report",
                    "expected_file": str(vep_path),
                })
                continue

            vep = add_variant_intervals(vep)
            vep["run"] = sample.run
            vep["sample_id"] = sample.sample_id
            vep["caller"] = caller
            vep["source_file"] = str(vep_path)
            vep["variant_row_id"] = [
                f"{sample.sample_id}|{caller}|row_{i}" for i in range(len(vep))
            ]

            bad_coords = vep[["variant_chrom", "variant_start", "variant_end"]].isna().any(axis=1)

            if bad_coords.any():
                problems.append({
                    "run": sample.run,
                    "sample_id": sample.sample_id,
                    "caller": caller,
                    "issue": f"{bad_coords.sum()} variants had unparseable coordinates",
                    "expected_file": str(vep_path),
                })

            overlap = vep.apply(label_one_variant, axis=1, bed_by_chrom=bed_by_chrom)
            all_variant_rows.append(pd.concat([vep, overlap], axis=1))

        except Exception as error:
            problems.append({
                "run": sample.run,
                "sample_id": sample.sample_id,
                "caller": caller,
                "issue": f"error: {error}",
                "expected_file": str(vep_path),
            })

print(f"Processed {len(metrics)} samples.")

problems_df = pd.DataFrame(problems)

if not all_variant_rows:
    problems_df.to_csv(
        output_dir / f"{output_prefix}_{run_tag}_missing_or_problem_files.csv",
        index=False,
    )
    raise RuntimeError("No variant rows were processed.")

per_variant = pd.concat(all_variant_rows, ignore_index=True)
per_variant["on_target"] = per_variant["on_target"].fillna(False).astype(bool)

print(f"Variants processed: {len(per_variant):,}")
print(f"Problems recorded: {len(problems_df):,}")

In [ ]:
problems_df

## Build QC summary tables

This section summarizes on-target and off-target variants by sample, caller, run, BED region, and gene.

In [ ]:
sample_summary = (
    per_variant
    .groupby(["run", "sample_id", "caller"], as_index=False)
    .agg(
        total_variants=("variant_row_id", "count"),
        on_target_variants=("on_target", "sum"),
    )
)

sample_summary["off_target_variants"] = (sample_summary["total_variants"] - sample_summary["on_target_variants"])

sample_summary["pct_on_target"] = (sample_summary["on_target_variants"] / sample_summary["total_variants"] * 100)

sample_summary["pct_off_target"] = (sample_summary["off_target_variants"] / sample_summary["total_variants"] * 100)

metrics_merge_cols = ["sample_id", "run"] + [col for col in metrics_qc_cols if col in metrics.columns]

sample_summary = sample_summary.merge(
    metrics[metrics_merge_cols].drop_duplicates(),
    on=["sample_id", "run"],
    how="left",
)

run_summary = (
    sample_summary
    .groupby(["run", "caller"], as_index=False)
    .agg(
        samples=("sample_id", "nunique"),
        total_variants=("total_variants", "sum"),
        on_target_variants=("on_target_variants", "sum"),
        off_target_variants=("off_target_variants", "sum"),
        mean_pct_on_target=("pct_on_target", "mean"),
        median_pct_on_target=("pct_on_target", "median"),
    )
)

run_summary["overall_pct_on_target"] = (run_summary["on_target_variants"] / run_summary["total_variants"] * 100)

run_summary["overall_pct_off_target"] = (run_summary["off_target_variants"] / run_summary["total_variants"] * 100)

off_target_variants = per_variant[~per_variant["on_target"]].copy()

on_target_by_bed_region = (
    per_variant[per_variant["on_target"]]
    .groupby(["run", "caller", "matched_bed_region"], as_index=False)
    .agg(
        on_target_variants=("variant_row_id", "count"),
        samples_with_variant=("sample_id", "nunique"),
    )
    .sort_values(["run", "caller", "on_target_variants"], ascending=[True, True, False])
)

on_off_by_vep_gene = (
    per_variant
    .groupby(["run", "caller", "Gene", "on_target"], as_index=False)
    .agg(
        variants=("variant_row_id", "count"),
        samples_with_variant=("sample_id", "nunique"),
    )
    .sort_values(["run", "caller", "Gene", "on_target"], ascending=[True, True, True, False])
)

bed_target_summary = (
    bed.assign(target_length=bed["bed_end"] - bed["bed_start"])
    .groupby("bed_region", as_index=False)
    .agg(
        num_bed_intervals=("bed_interval_id", "count"),
        total_target_bases=("target_length", "sum"),
    )
    .sort_values("total_target_bases", ascending=False)
)

run_summary

## On-target percentage by run and caller

Compare the mean sample-level on-target percentage across runs and callers.

In [ ]:
run_order = metrics["run"].drop_duplicates().tolist()
x = np.arange(len(run_order))
width = 0.8 / len(callers)

plt.figure(figsize=(9, 5))

for i, caller in enumerate(callers):
    values = (
        run_summary[run_summary["caller"] == caller]
        .set_index("run")
        .reindex(run_order)["mean_pct_on_target"]
    )
    plt.bar(x + i * width, values, width, label=caller)

plt.xticks(x + width * (len(callers) - 1) / 2, run_order)
plt.ylim(0, 100)
plt.xlabel("Run")
plt.ylabel("Mean on-target variants (%)")
plt.title("Mean on-target variant percentage by run and caller")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.gca().set_axisbelow(True)

figure_path = output_dir / f"{output_prefix}_{run_tag}_mean_pct_on_target_by_run_and_caller.png"

plt.tight_layout()
plt.savefig(figure_path, dpi=300)
plt.show()
plt.close()

## Save QC outputs

The full per-variant target-status table is saved as CSV.

The Excel workbook contains summary sheets and, if small enough for Excel, the per-variant table.

A smaller Excel preview is also saved using only the first 10 processed samples so it can be opened easily.

## Save separate outputs for each run

Although all selected runs are processed together, the resulting tables are split by run before saving. Each run receives its own full per-variant CSV, off-target CSV, summary workbook, and preview workbook.

In [ ]:
def rows_for_run(table, run):
    return table[table["run"].eq(run)] if "run" in table.columns else table

for run in sorted(metrics["run"].dropna().unique()):
    run_variants = per_variant[per_variant["run"].eq(run)].copy()
    run_off_target = run_variants[~run_variants["on_target"]].copy()

    preview_samples = run_variants["sample_id"].drop_duplicates().head(excel_preview_n_samples)
    run_preview = run_variants[run_variants["sample_id"].isin(preview_samples)]
    run_off_target_preview = run_preview[~run_preview["on_target"]]

    full_csv = output_dir / f"{output_prefix}_{run}_per_variant_target_status_FULL.csv"
    off_target_csv = output_dir / f"{output_prefix}_{run}_off_target_variants.csv"
    summary_xlsx = output_dir / f"{output_prefix}_{run}_on_off_target_summary.xlsx"
    preview_xlsx = output_dir / f"{output_prefix}_{run}_per_variant_target_status_preview_{excel_preview_n_samples}_samples.xlsx"

    run_variants.to_csv(full_csv, index=False)
    run_off_target.to_csv(off_target_csv, index=False)

    with pd.ExcelWriter(summary_xlsx) as writer:
        rows_for_run(sample_summary, run).to_excel(writer, sheet_name="sample_caller_summary", index=False)
        rows_for_run(run_summary, run).to_excel(writer, sheet_name="run_caller_summary", index=False)
        rows_for_run(on_target_by_bed_region, run).to_excel(writer, sheet_name="on_target_by_bed_region", index=False)
        rows_for_run(on_off_by_vep_gene, run).to_excel(writer, sheet_name="on_off_by_vep_gene", index=False)
        rows_for_run(bed_target_summary, run).to_excel(writer, sheet_name="bed_target_summary", index=False)
        rows_for_run(problems_df, run).to_excel(writer, sheet_name="problem_files", index=False)

    with pd.ExcelWriter(preview_xlsx) as writer:
        run_preview.to_excel(writer, sheet_name="per_variant_preview", index=False)
        run_off_target_preview.to_excel(writer, sheet_name="off_target_preview", index=False)

    print(f"{run}: {len(run_variants):,} variants saved")